In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/infomediaupdates/sample/sample_submission.csv
/kaggle/input/datasets/infomediaupdates/sample/train.csv
/kaggle/input/datasets/infomediaupdates/sample/test.csv


In [4]:
# This Python 3 environment comes with many helpful analytics libraries installed
import numpy as np
import pandas as pd
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/infomediaupdates/sample/sample_submission.csv
/kaggle/input/datasets/infomediaupdates/sample/train.csv
/kaggle/input/datasets/infomediaupdates/sample/test.csv


In [5]:
import warnings
warnings.filterwarnings("ignore")

DATA_DIR = "/kaggle/input/datasets/infomediaupdates/sample"

train = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")
sample = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")

print(train.shape, test.shape)
train.head()

(1000, 12) (200, 11)


,id,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,feature_10,target
0,1,-3.473887,-1.357287,-1.787686,-2.559501,-1.598815,1.112116,-0.859805,0.401226,0.328429,-0.850505,1
1,2,-2.589476,-0.458541,-1.800683,-1.332204,-2.640357,3.353933,-0.617607,0.455737,-0.023545,-2.324505,1
2,3,3.257358,-1.024133,2.314914,3.956688,-1.478358,-4.327307,1.053220,-1.660185,1.737734,3.349098,0
3,4,0.026609,-0.765901,-0.210356,0.639859,-0.256846,1.702046,-0.869473,-0.139467,-0.174024,-1.373688,1
4,5,1.865959,-1.563378,0.844567,3.217234,-0.341449,-0.774112,1.039728,0.705972,-1.587741,-0.182342,0


In [6]:
print(train.info())
print(train.isnull().sum())
print(train.describe())

# Set your target column name here
TARGET = "target"   # <-- change to your actual target column name
print(train[TARGET].value_counts())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 12 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   id          1000 non-null   int64  
 1   feature_1   1000 non-null   float64
 2   feature_2   1000 non-null   float64
 3   feature_3   1000 non-null   float64
 4   feature_4   1000 non-null   float64
 5   feature_5   1000 non-null   float64
 6   feature_6   1000 non-null   float64
 7   feature_7   1000 non-null   float64
 8   feature_8   1000 non-null   float64
 9   feature_9   1000 non-null   float64
 10  feature_10  1000 non-null   float64
 11  target      1000 non-null   int64  
dtypes: float64(10), int64(2)
memory usage: 93.9 KB
None
id            0
feature_1     0
feature_2     0
feature_3     0
feature_4     0
feature_5     0
feature_6     0
feature_7     0
feature_8     0
feature_9     0
feature_10    0
target        0
dtype: int64
                id    feature_1    feature_2    feature_3 

In [7]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

X = train.drop(columns=[TARGET])
y = train[TARGET]
X_test = test.copy()

cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = X.select_dtypes(exclude=["object", "category"]).columns.tolist()

print("Categorical:", cat_cols)
print("Numerical:", num_cols)

for c in num_cols:
    X[c] = X[c].fillna(X[c].median())
    X_test[c] = X_test[c].fillna(X[c].median())

for c in cat_cols:
    X[c] = X[c].fillna("missing")
    X_test[c] = X_test[c].fillna("missing")

encoders = {}
for c in cat_cols:
    le = LabelEncoder()
    combined = pd.concat([X[c], X_test[c]], axis=0).astype(str)
    le.fit(combined)
    X[c] = le.transform(X[c].astype(str))
    X_test[c] = le.transform(X_test[c].astype(str))
    encoders[c] = le

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42,
    stratify=y if y.nunique() < 20 else None
)

Categorical: []
Numerical: ['id', 'feature_1', 'feature_2', 'feature_3', 'feature_4', 'feature_5', 'feature_6', 'feature_7', 'feature_8', 'feature_9', 'feature_10']


In [8]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.metrics import roc_auc_score

ada = AdaBoostClassifier(
    n_estimators=200,
    learning_rate=0.5,
    random_state=42
)
ada.fit(X_train, y_train)

ada_val_pred = ada.predict_proba(X_val)[:, 1]
print("AdaBoost AUC:", roc_auc_score(y_val, ada_val_pred))

AdaBoost AUC: 0.8908


In [9]:
import xgboost as xgb

xgb_model = xgb.XGBClassifier(
    n_estimators=2000,
    learning_rate=0.03,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="auc",
    early_stopping_rounds=100,
    random_state=42,
    tree_method="hist"
)

xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=100
)

xgb_val_pred = xgb_model.predict_proba(X_val)[:, 1]
print("XGBoost AUC:", roc_auc_score(y_val, xgb_val_pred))
print("Best iteration:", xgb_model.best_iteration)

[0]	validation_0-auc:0.84665
[100]	validation_0-auc:0.95150
[200]	validation_0-auc:0.95660
[300]	validation_0-auc:0.95880
[400]	validation_0-auc:0.96130
[500]	validation_0-auc:0.96240
[600]	validation_0-auc:0.96180
[642]	validation_0-auc:0.96110
XGBoost AUC: 0.9625999999999999
Best iteration: 542


In [10]:
import lightgbm as lgb

lgb_model = lgb.LGBMClassifier(
    n_estimators=2000,
    learning_rate=0.03,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

lgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric="auc",
    callbacks=[lgb.early_stopping(stopping_rounds=100), lgb.log_evaluation(100)]
)

lgb_val_pred = lgb_model.predict_proba(X_val)[:, 1]
print("LightGBM AUC:", roc_auc_score(y_val, lgb_val_pred))
print("Best iteration:", lgb_model.best_iteration_)

[LightGBM] [Info] Number of positive: 400, number of negative: 400
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000922 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2805
[LightGBM] [Info] Number of data points in the train set: 800, number of used features: 11
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
Training until validation scores don't improve for 100 rounds
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive ga

In [11]:
from catboost import CatBoostClassifier

cat_model = CatBoostClassifier(
    iterations=2000,
    learning_rate=0.03,
    depth=6,
    eval_metric="AUC",
    random_seed=42,
    verbose=100,
    early_stopping_rounds=100
)

cat_model.fit(
    X_train, y_train,
    eval_set=(X_val, y_val),
    cat_features=cat_cols
)

cat_val_pred = cat_model.predict_proba(X_val)[:, 1]
print("CatBoost AUC:", roc_auc_score(y_val, cat_val_pred))

0:	test: 0.7492500	best: 0.7492500 (0)	total: 54.6ms	remaining: 1m 49s
100:	test: 0.9641000	best: 0.9646000 (98)	total: 292ms	remaining: 5.49s
200:	test: 0.9699000	best: 0.9705000 (189)	total: 535ms	remaining: 4.79s
300:	test: 0.9733000	best: 0.9735000 (279)	total: 774ms	remaining: 4.37s
400:	test: 0.9740000	best: 0.9744000 (330)	total: 1.01s	remaining: 4.03s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.9744
bestIteration = 330

Shrink model to first 331 iterations.
CatBoost AUC: 0.9743999999999999


In [12]:
results = {
    "AdaBoost": roc_auc_score(y_val, ada_val_pred),
    "XGBoost": roc_auc_score(y_val, xgb_val_pred),
    "LightGBM": roc_auc_score(y_val, lgb_val_pred),
    "CatBoost": roc_auc_score(y_val, cat_val_pred),
}

for name, score in sorted(results.items(), key=lambda x: -x[1]):
    print(f"{name}: {score:.5f}")

CatBoost: 0.97440
XGBoost: 0.96260
LightGBM: 0.95780
AdaBoost: 0.89080


In [13]:
best_model = lgb_model  # swap to whichever scored highest above

test_pred = best_model.predict_proba(X_test)[:, 1]

sample[TARGET] = test_pred
sample.to_csv("submission.csv", index=False)
sample.head()

,id,target
0,1001,0.981797
1,1002,0.510341
2,1003,0.002750
3,1004,0.977070
4,1005,0.457185
